In [0]:
%pip install -U \
  databricks-langchain==0.16.1 \
  langchain==1.2.10 \
  mlflow>=3.0 \
  langgraph \
  nest_asyncio

# langgraph==1.0.x declares a floor of langgraph-prebuilt>=1.0.7, but every langgraph-prebuilt
# release in that range imports `ExecutionInfo`/`ServerInfo` from langgraph.runtime — symbols
# that don't exist until langgraph 1.1+ (which langchain==1.2.10 doesn't yet allow). Force the
# last prebuilt release that predates that import so `from langgraph.prebuilt import ToolNode`
# doesn't crash with `ImportError: cannot import name 'ExecutionInfo'`.
%pip install -q "langgraph-prebuilt==1.0.1" --no-deps --force-reinstall

try:
    dbutils.library.restartPython()
except:
    pass

In [0]:
import asyncio
from typing import Annotated, Sequence, TypedDict

import mlflow
import nest_asyncio
from databricks.sdk import WorkspaceClient
from databricks_langchain import (
    ChatDatabricks,
    DatabricksMCPServer,
    DatabricksMultiServerMCPClient,
    MCPServer,
)
from langchain_core.messages import AIMessage, AnyMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

nest_asyncio.apply()
mlflow.langchain.autolog()

import os

# ── Workspace + schema ────────────────────────────────────────────────────────
w        = WorkspaceClient()
username = w.current_user.me().user_name.split("@")[0].replace(".", "_").replace("-", "_")
catalog  = "bootcamp_students"
schema   = f"maintops"

print(f"User:    {username}")
print(f"Schema:  {catalog}.{schema}")


In [0]:
mcp_servers = [
    DatabricksMCPServer.from_vector_search(
        catalog=catalog, schema=schema,
        index_name="faq_index",
        name="maintops-faq-vector-search", workspace_client=w,
    ),
]

mcp_client = DatabricksMultiServerMCPClient(mcp_servers)

print(f"MCP client ready — {len(mcp_servers)} servers")

In [0]:
async def _load_tools():
    try:
        return await mcp_client.get_tools()
    except Exception as e:
        print(f"Bulk tool load failed ({type(e).__name__}), trying servers individually...")
        all_tools = []
        for server in mcp_servers:
            try:
                client = DatabricksMultiServerMCPClient([server])
                tools = await client.get_tools()
                all_tools.extend(tools)
                print(f"  ✓ {server.name}: {len(tools)} tool(s)")
            except Exception as se:
                print(f"  ✗ {server.name}: SKIPPED — {se}")
        return all_tools

mcp_tools = asyncio.run(_load_tools())

print(f"Total tools loaded: {len(mcp_tools)}\n")
for tool in mcp_tools:
    desc = tool.description or "\u2014"
    print(f"  [{tool.name}]")
    print(f"    {desc[:120]}{'...' if len(desc) > 120 else ''}")
    print()


In [0]:
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-1-8b-instruct",
    temperature=0.1,
)

SYSTEM_PROMPT = """You are Manny, the MaintOps AI assistant.

You serve unregistered users who want to learn about MaintOps, understand how the platform works, and determine whether it can help with their maintenance needs.

Always use the available knowledge base to ground your answers about MaintOps. Never make up facts.

When asked about MaintOps, its functionality, registration, handyman recommendations, supported maintenance problems, the client or handyman experience, incidents, ratings, feedback, pricing, availability, or other platform-related topics, use the MaintOps guideline knowledge base.

Answer questions clearly and concisely using user-friendly language.

If the requested information is not available in the knowledge base, say that it is not currently specified rather than making assumptions.

MaintOps is not an emergency service. If a user describes an immediate danger to people or property, advise them to contact the appropriate emergency or specialist service.
"""

class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]


def build_agent(tools):
    """Build a LangGraph tool-calling agent backed by MCP tools."""
    model = llm.bind_tools(tools)
    preprocessor = RunnableLambda(
        lambda state: [{"role": "system", "content": SYSTEM_PROMPT}] + list(state["messages"])
    )

    def should_continue(state: AgentState):
        last = state["messages"][-1]
        return "continue" if isinstance(last, AIMessage) and last.tool_calls else "end"

    def call_model(state: AgentState, config: RunnableConfig):
        return {"messages": [(preprocessor | model).invoke(state, config)]}

    g = StateGraph(AgentState)
    g.add_node("agent", RunnableLambda(call_model))
    g.add_node("tools", ToolNode(tools))
    g.set_entry_point("agent")
    g.add_conditional_edges("agent", should_continue, {"continue": "tools", "end": END})
    g.add_edge("tools", "agent")
    return g.compile()


manny = build_agent(mcp_tools)
print("Manny agent ready!")
print(f"Tools bound: {[t.name for t in mcp_tools]}")

In [0]:
def ask_manny(question: str) -> str:
    """Invoke Zachy and print the final answer."""
    print(f"User:  {question}")
    print()
    result = asyncio.run(manny.ainvoke({"messages": [{"role": "user", "content": question}]}))
    answer = result["messages"][-1].content
    print(f"Zachy: {answer}")
    return answer

In [0]:
ask_manny("Can I choose my handyman?")

In [0]:
ask_manny("Do I need an account?")

In [0]:
ask_manny("How do you decide who to recommend?")

In [0]:
ask_manny("What if I have an emergency?")

In [0]:
ask_manny("Do you always choose the closest person?")

In [0]:
ask_manny("Can electricians register?")

In [0]:
ask_manny("How much does the service cost?")